# Lab Day 10 trên Kaggle: chạy toàn bộ bằng một nút

**Trước khi chạy** (bảng bên phải → *Session options*):
1. **Accelerator** = `GPU T4 x2`.
2. **Internet** = On (cần xác minh số điện thoại cho tài khoản Kaggle).

**Chạy thử nhanh (≈30 phút):** để `SMOKE_TEST = True`, bấm *Run All*, chờ dòng `XONG, không có lỗi`.
**Chạy thật (≈5–7 giờ):** đổi thành `SMOKE_TEST = False`, bấm *Save Version* → *Save & Run All (Commit)*.
Xong thì vào tab *Output* của phiên bản đó, tải file **`lab10_outputs.zip`** và gửi lại.

In [ ]:
SMOKE_TEST = True          # True = chạy thử nhanh; False = chạy thật để nộp bài
RUN_E5 = True              # bonus GRPO (+10 điểm); đặt False nếu muốn tiết kiệm giờ GPU
REPO_URL = "https://github.com/tubepvhat1604-bot/K4-Track-4-Day10-VisionLanguageModel.git"
BRANCH = "claude/project-thread-llnd8q"

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("lab"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "lab"], check=True)
os.chdir("/kaggle/working/lab")
subprocess.run(["git", "log", "--oneline", "-1"])
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Tìm dataset: Kaggle (kagglehub, không cần khoá) → /kaggle/input → Kaggle CLI (Colab Secrets).
import glob, os, subprocess

def find_dataset():
    try:
        import kagglehub
        p = kagglehub.dataset_download("lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes")
        if glob.glob(os.path.join(p, "**", "labels-YOLO"), recursive=True):
            return p
    except Exception as e:
        print("kagglehub không dùng được:", e)
    hits = glob.glob("/kaggle/input/**/labels-YOLO", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if not os.path.exists("data/splits/meta.json"):
    src = find_dataset()
    if src:
        subprocess.run(["python", "scripts/prepare_data.py", "--source", src], check=True)
    else:   # Colab: KAGGLE_USERNAME / KAGGLE_KEY trong Colab Secrets, KHÔNG viết khoá vào notebook
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        subprocess.run(["python", "scripts/prepare_data.py", "--kaggle"], check=True)
!cat data/splits/meta.json | head -5

In [ ]:
# Nếu phiên trước bị ngắt giữa chừng: thêm Output của phiên trước làm Input, ô này chép lại để chạy tiếp.
import glob, shutil
for prev in glob.glob("/kaggle/input/**/lab/outputs/evidence", recursive=True):
    base = os.path.dirname(os.path.dirname(prev))
    for d in ("outputs", "adapters"):
        if os.path.exists(os.path.join(base, d)):
            shutil.copytree(os.path.join(base, d), d, dirs_exist_ok=True)
    print("đã chép kết quả cũ từ", base)

In [ ]:
steps = ["all"] if RUN_E5 else ["stats", "e0", "e1", "e2", "e3", "e4", "extras"]
cmd = " ".join(steps) + (" --smoke" if SMOKE_TEST else "")
!python -m lab10.run {cmd} 2>&1 | grep --line-buffered -v "Loading weights"

In [ ]:
# Gom kết quả để gửi lại (không chứa trọng số, không chứa ảnh dataset)
import shutil, json
out = "outputs_smoke" if SMOKE_TEST else "outputs"
shutil.copy("data/splits/meta.json", f"{out}/meta.json")
shutil.make_archive("/kaggle/working/lab10_outputs" + ("_smoke" if SMOKE_TEST else ""), "zip", out)
for f in sorted(glob.glob(f"{out}/evidence/*.json")):
    d = json.load(open(f))
    print(os.path.basename(f), "F1=%.4f" % d["micro"]["f1"], "CI95=", [round(x, 4) for x in d["f1_ci95"]],
          "parse_valid=%.3f" % d["parse_valid_rate"], d.get("meta", {}))
print("\nFile để tải về:", "/kaggle/working/lab10_outputs" + ("_smoke" if SMOKE_TEST else "") + ".zip")